In [1]:
import os
import sys
import scipy
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import uuid
import multiprocessing as mp
from multiprocessing import Pool
sc._settings.ScanpyConfig.n_jobs=24
sc.settings.verbosity = 1

In [2]:
sc.logging.print_header()

Package,Version
scipy,1.15.2
numpy,2.2.5
pandas,2.2.3
scanpy,1.11.1
anndata,0.11.4
tqdm,4.66.5
Component,Info
Python,"3.12.6 | packaged by conda-forge | (main, Sep 30 2024, 18:08:52) [GCC 13.3.0]"
OS,Linux-5.15.0-117-generic-x86_64-with-glibc2.31
CPU,"96 logical CPU cores, x86_64"


## make_anndata, make_anndata_withoutBCR, make_anndata_withoutTCR

In [3]:
def make_anndata(celltype):
    global adata_raw
    global adt
    global obj_path
    indices = pd.read_csv(f'{obj_path}{celltype}/R2_indices_{celltype}.csv',index_col=0)
    if len(indices.index)>2000:
        adata = adata_raw[indices.index].to_memory()
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata,flavor='seurat_v3',n_top_genes=2000,layer="counts",
                                subset=True)
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")
    else:
        adata = adata_raw[indices.index].to_memory()
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        adata.var['highly_variable']=True
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")

In [4]:
def make_anndata_withoutBCR(celltype):
    global adata_raw
    global adt
    global obj_path
    indices = pd.read_csv(f'{obj_path}{celltype}/R2_indices_{celltype}.csv',index_col=0)
    if len(indices.index)>2000:
        adata = adata_raw[indices.index]
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata,flavor='seurat_v3',n_top_genes=2000,layer="counts",
                                subset=True)

        #Generate new highly_variable assignments except BCR
        bcr = pd.read_table('BCR_gene.txt',header=None)
        for i in range(1,len(bcr.values)):
            ind = adata.var_names.isin(bcr.values[i])
            adata.var.loc[ind,'highly_variable'] = False
        adata = adata[:, adata.var['highly_variable']].copy()
    
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")
    else:
        adata = adata_raw[indices.index]
        
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        adata.var['highly_variable']=True
        #Generate new highly_variable assignments except BCR
        bcr = pd.read_table('BCR_gene.txt',header=None)
        for i in range(1,len(bcr.values)):
            ind = adata.var_names.isin(bcr.values[i])
            adata.var.loc[ind,'highly_variable'] = False
        adata = adata[:, adata.var['highly_variable']].copy()
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")

In [5]:
def make_anndata_withoutTCR(celltype):
    global adata_raw
    global adt
    global obj_path
    indices = pd.read_csv(f'{obj_path}{celltype}/R2_indices_{celltype}.csv',index_col=0)
    if len(indices.index)>2000:
        adata = adata_raw[indices.index].to_memory()
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata,flavor='seurat_v3',n_top_genes=2000,layer="counts",
                                subset=True)

        #Generate new highly_variable assignments except TCR
        tcr = pd.read_table('TCR_gene.txt',header=None)
        for i in range(1,len(tcr.values)):
            ind = adata.var_names.isin(tcr.values[i])
            adata.var.loc[ind,'highly_variable'] = False
        adata = adata[:, adata.var['highly_variable']].copy()

    
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")
    else:
        adata = adata_raw[indices.index].to_memory()
        ##　Save scRNA count
        adata.write(f"{obj_path}{celltype}/{celltype}_count_scRNA.h5ad",compression="gzip")
        
        adata.layers["counts"] = adata.X.copy()
        adata.var['highly_variable']=True
        #Generate new highly_variable assignments except TCR
        tcr = pd.read_table('TCR_gene.txt',header=None)
        for i in range(1,len(tcr.values)):
            ind = adata.var_names.isin(tcr.values[i])
            adata.var.loc[ind,'highly_variable'] = False
        adata = adata[:, adata.var['highly_variable']].copy()
        ##　Save scRNA for TOTALVI
        adata.write(f"{obj_path}{celltype}/{celltype}_preprocess_scRNA.h5ad",compression="gzip")
        
        
        ##　Save scADT for TOTALVI
        adt_sub = adt[indices.index].copy()
        adt_sub.write(f"{obj_path}{celltype}/{celltype}_preprocess_scADT.h5ad",compression="gzip")

        print(f"{celltype},{adt_sub.n_obs},{len(adata.var['highly_variable'])} Done!")

# 1. Got HVG and TOTALVI data for L1 refine-low_nCount_RNA

In [11]:
dataset = "low_nCount_RNA"

In [12]:
obj_path = f'/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R2/'

In [13]:
adata_raw = sc.read_h5ad(f'/home/liyanguo/MyImmuCell/02_Read_QC/scRNA_MyImmuCell_{dataset}.h5ad')#600Gb memory, if we add backed='r', use adata = adata_raw[indices.index].to_memory()

In [14]:
adt = sc.read_h5ad(f'/home/liyanguo/MyImmuCell/04_MyImmuCell_TOTALVI/scADT_MyImmuCell_{dataset}.h5ad')

In [15]:
celltypes=['Basophil','CEACAM8_Neg_Neutrophil','Platelet']

In [16]:
for i in celltypes:
    make_anndata(i)

/tmp/ipykernel_2703473/3296954346.py:11: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  adata.layers["counts"] = adata.X.copy()


Basophil,51145,2000 Done!


/tmp/ipykernel_2703473/3296954346.py:11: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  adata.layers["counts"] = adata.X.copy()


CEACAM8_Neg_Neutrophil,28681377,2000 Done!


/tmp/ipykernel_2703473/3296954346.py:11: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  adata.layers["counts"] = adata.X.copy()


Platelet,9308,2000 Done!


# 1. Got HVG and TOTALVI data for L1 refine-high_nCount_RNA

In [7]:
dataset = "high_nCount_RNA"

In [8]:
obj_path = f'/home/liyanguo/MyImmuCell/05_MyImmuCell_subpopulation/Level2_Refine_R2/'

In [9]:
#Read in to 1000Gb memory
# if we add backed='r', use adata = adata_raw[indices.index].to_memory(), to process few cell data

adata_raw = sc.read_h5ad(f'/home/liyanguo/MyImmuCell/02_Read_QC/scRNA_MyImmuCell_{dataset}.h5ad',backed='r')

In [10]:
adata_raw.isbacked

True

In [11]:
adt = sc.read_h5ad(f'/home/liyanguo/MyImmuCell/04_MyImmuCell_TOTALVI/scADT_MyImmuCell_{dataset}.h5ad')

## Normal process

In [14]:
celltypes=['HSPC',
           'DC',
           'Non_NK_ILC',
           'CEACAM8_Pos_Neutrophil',
           'gdT',
           'MAIT',
           'iNKT',
           'Mast',
           'Monocyte',
          ]

In [18]:
celltypes=['HSPC',
           'Non_NK_ILC'
          ]

In [19]:
for i in celltypes:
    make_anndata(i)

HSPC,18927,2000 Done!
Non_NK_ILC,17562,2000 Done!


In [10]:
celltypes=[
           'NK',
          ]

In [11]:
for i in celltypes:
    make_anndata(i)

NK,4925060,2000 Done!


## Remove BCR

In [10]:
celltypes=['NaiveB',
            'MemoryB',
            'Plasma']

In [11]:
for i in celltypes:
    make_anndata_withoutBCR(i)

NaiveB,1232786,1804 Done!
MemoryB,694593,1798 Done!
Plasma,48633,1798 Done!


## Remove TCR

In [27]:
celltypes=['CytotoxicCD4',
           'NaiveCD8',
           'Treg',
           'TemCD8',
           'CD4_helper_memory',
           'CD8Tcm',
           'NaiveCD4'
          ]

In [12]:
celltypes=['Treg',
           'CD4_helper_memory',
           'NaiveCD4'
          ]

In [13]:
for i in celltypes:
    make_anndata_withoutTCR(i)

Treg,288057,1890 Done!
CD4_helper_memory,4547781,1891 Done!
NaiveCD4,2798231,1891 Done!
